In [ ]:
# ===================== ONE-CELL: Unified Hybrid Brain Tumor Pipeline (Leakage-free MD5 split + Detection→Classification + 28 variants + External TEST2 eval-only + EXPORT ALL + Efficiency/Radar) =====================
# Copy-paste as ONE Jupyter cell. Runs on Windows.
#
# What you get (in OUT_DIR):
#  - ablation_summary.csv                         (VAL/TEST/TEST2 metrics)
#  - *_test_predictions.csv  for ALL 28 variants  (path,y_true,y_pred,probabilities)
#  - *_test_cm.csv           for ALL 28 variants  (confusion matrix)
#  - *_test2_predictions.csv (if TEST2 exists)
#  - efficiency_summary.csv                       (train time, params, latency, efficiency index)
#  - efficiency_radar.png                         (radar like your figure)
#  - class_names.txt
#
# IMPORTANT:
#  1) Update TRAIN_ROOT and TEST2_ROOT paths below to your folders.
#  2) First run may be slow because it extracts & caches backbone features (backbone cost is excluded from efficiency).
# =========================================================================================================================================================================

# ---------------- Auto-install (Windows/Jupyter-safe) ----------------
import os, sys, subprocess, importlib
os.makedirs("./_pip_cache", exist_ok=True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # determinism hint (CUDA)

def _ensure(pkgs):
    modmap = {"scikit-learn":"sklearn", "scikit-image":"skimage"}
    for p in pkgs:
        try:
            importlib.import_module(modmap.get(p,p))
        except Exception:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--cache-dir", "./_pip_cache", p])

_ensure([
    "timm>=1.0.20",
    "torchvision",
    "tqdm",
    "scikit-learn",
    "numpy",
    "scipy",
    "pillow",
    "tabulate",
    "scikit-image",
    "matplotlib",
])

# ---------------- Imports ----------------
import math, random, shutil, warnings, json, csv, hashlib, time
import numpy as np
from pathlib import Path
from collections import Counter

import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import AdamW
try:
    from torch.amp import GradScaler, autocast
except Exception:
    from torch.cuda.amp import GradScaler, autocast

from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.datasets.folder import default_loader
from PIL import Image
try:
    RESAMPLE_BILINEAR = Image.Resampling.BILINEAR
except AttributeError:
    RESAMPLE_BILINEAR = Image.BILINEAR

from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.linear_model import LogisticRegression
from tabulate import tabulate

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

# ============================= USER CONFIG ===================================================
# ---- TRAIN dataset (ONLY this is used for training) ----
TRAIN_NAME = "jan18_final_nodup"
TRAIN_ROOT = r"C:\Users\cse.majib\Desktop\Data sets\New folder\New folder_CLEAN_NO_LEAK"   # <<< CHANGE if needed
OUT_DIR    = r"./runs_unified_jan18_final_nodup"
CACHE_DIR  = r"./feature_cache_unified_jan18_final_nodup"

# ---- EXTERNAL TEST2 dataset (EVAL ONLY; NEVER split; NEVER train on it) ----
USE_TEST2 = True
TEST2_NAME  = "jan18_local_TEST2"
TEST2_ROOT  = r"C:\Users\cse.majib\Desktop\Data sets\18 jan data\local"                    # <<< CHANGE if needed
TEST2_CACHE = r"./feature_cache_unified_jan18_local_TEST2"

# Splits for TRAIN_ROOT only (if it is not already split into train/val/test)
SEED = 2025
VAL_RATIO, TEST_RATIO = 0.15, 0.10

# Feature extraction (backbone)
EXTRACT_BATCH_SIZE = 8
TTA_MODE_VALTEST = "flip"      # {"none","flip","five"} applied on VAL/TEST/TEST2 (not train)
ALLOW_FLIP = True

# Head training on cached features
EPOCHS = 40
LR = 2e-4
WEIGHT_DECAY = 5e-5
WARMUP_FRAC = 0.05
BATCH_SIZE = 32
LABEL_SMOOTH = 0.10

USE_AMP = True
USE_EMA = True
USE_EMA_FOR_SELECTION = True
EMA_DECAY = 0.999
MIXUP_ALPHA = 0.4

USE_ZSCORE = True
USE_FINGERPRINT = True
SAVE_CACHE_FP16 = True
LOAD_CAST_FLOAT32 = True

ATTN_FUSE_DIM = 1024
ENSEMBLE_ACTS = ["mish","silu","gelu"]     # for *_ens variants
SINGLE_ACT    = "mish"                      # for *_single variants

# Detection→Classification (NONE / notumor class is detected by name)
DETECT_THRESH = 0.50  # tumor if (1 - P_none) >= DETECT_THRESH

# Backbones (full-image only; backbone cost excluded from efficiency)
INCLUDE_DINOV3_IF_AVAILABLE = False  # set True if you want it (often heavy)
CAND_MODELS = [
    "convnext_base",
    "tf_efficientnetv2_m_in21ft1k",
    "densenet201",
    "swin_small_patch4_window7_224",
    "vit_base_patch16_224",
]

DO_TEMPERATURE_SCALING = True

# ============================= ENV/SEEDS ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(USE_AMP and torch.cuda.is_available())
_amp_dtype = torch.float16
NUM_WORKERS = 0
PIN_MEMORY = bool(torch.cuda.is_available())
DETERMINISTIC = True

def set_all_seeds(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if DETERMINISTIC and torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try: torch.use_deterministic_algorithms(True)
        except Exception: pass

if torch.cuda.is_available():
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass

print(f"Device: {device} | AMP: {USE_AMP}")

# ============================= MD5 leakage-free split ========================================
def md5_file(path, chunk_size=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def _safe_move_unique(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        try:
            shutil.move(src, dst); return
        except Exception:
            shutil.copy2(src, dst); os.remove(src); return
    base, ext = os.path.splitext(dst)
    for k in range(1, 10_000):
        cand = f"{base}__dup{k}{ext}"
        if not os.path.exists(cand):
            try:
                shutil.move(src, cand); return
            except Exception:
                shutil.copy2(src, cand); os.remove(src); return

def verify_no_overlap_by_md5(root_dir):
    def _collect_paths(split):
        split_dir = os.path.join(root_dir, split)
        paths = []
        if not os.path.isdir(split_dir): return paths
        for cls in os.listdir(split_dir):
            cdir = os.path.join(split_dir, cls)
            if not os.path.isdir(cdir): continue
            for fn in os.listdir(cdir):
                p = os.path.join(cdir, fn)
                if os.path.isfile(p): paths.append(p)
        return paths

    md5_sets = {}
    for split in ["train","val","test"]:
        paths = _collect_paths(split)
        s = set()
        for p in tqdm(paths, desc=f"[MD5 verify] {split}", ncols=100):
            s.add(md5_file(p))
        md5_sets[split] = s

    inter_tv = md5_sets["train"] & md5_sets["val"]
    inter_tt = md5_sets["train"] & md5_sets["test"]
    inter_vt = md5_sets["val"]   & md5_sets["test"]
    print(f"[MD5 verify] unique(train)={len(md5_sets['train'])} unique(val)={len(md5_sets['val'])} unique(test)={len(md5_sets['test'])}")
    print(f"[MD5 verify] overlaps: train∩val={len(inter_tv)}, train∩test={len(inter_tt)}, val∩test={len(inter_vt)}")
    assert len(inter_tv)==0 and len(inter_tt)==0 and len(inter_vt)==0, "LEAKAGE FOUND: MD5 overlap exists across splits!"

def split_data_into_folders_md5_grouped(root_dir, val_ratio=0.15, test_ratio=0.10, seed=SEED):
    # If already split, verify and exit.
    if all(os.path.isdir(os.path.join(root_dir, p)) for p in ["train","val","test"]):
        print("train/val/test already exist. Verifying MD5 disjointness...")
        verify_no_overlap_by_md5(root_dir)
        return

    print("Creating stratified train/val/test split (MD5-grouped, leakage-free)...")
    temp_dataset = datasets.ImageFolder(root_dir)
    samples = temp_dataset.samples  # [(path,label),...]

    md5_cache_path = os.path.join(root_dir, "_md5_cache.json")
    md5_map = {}
    if os.path.exists(md5_cache_path):
        try:
            with open(md5_cache_path, "r") as f:
                md5_map = json.load(f)
        except Exception:
            md5_map = {}

    md5_list = []
    for p, _ in tqdm(samples, desc="[MD5] hashing", ncols=100):
        if p in md5_map:
            md = md5_map[p]
        else:
            md = md5_file(p)
            md5_map[p] = md
        md5_list.append(md)

    try:
        with open(md5_cache_path, "w") as f:
            json.dump(md5_map, f)
    except Exception:
        pass

    # group duplicates by MD5
    groups = {}
    for (p, y), md in zip(samples, md5_list):
        groups.setdefault(md, []).append((p, y))

    group_ids = list(groups.keys())
    group_labels = []
    for gid in group_ids:
        ys = [y for _, y in groups[gid]]
        group_labels.append(Counter(ys).most_common(1)[0][0])

    temp_ratio = val_ratio + test_ratio
    g_tr, g_vt, _, y_vt = train_test_split(
        group_ids, group_labels,
        test_size=temp_ratio,
        stratify=group_labels,
        random_state=seed
    )
    rel_test = test_ratio / temp_ratio
    g_va, g_te, _, _ = train_test_split(
        g_vt, y_vt,
        test_size=rel_test,
        stratify=y_vt,
        random_state=seed
    )

    split_map = {gid:"train" for gid in g_tr}
    split_map.update({gid:"val" for gid in g_va})
    split_map.update({gid:"test" for gid in g_te})

    for gid in tqdm(group_ids, desc="[MOVE] grouped duplicates", ncols=100):
        split_name = split_map[gid]
        for p, _ in groups[gid]:
            cls = os.path.basename(os.path.dirname(p))
            dst = os.path.join(root_dir, split_name, cls, os.path.basename(p))
            _safe_move_unique(p, dst)

    print("Split complete. Verifying MD5 disjointness...")
    verify_no_overlap_by_md5(root_dir)

# ============================= Dataset wrappers ==============================================
class RawImageFolder(datasets.ImageFolder):
    def __getitem__(self, index):
        path, target = self.samples[index]
        img = self.loader(path)
        if img.mode != "RGB": img = img.convert("RGB")
        return img, target, path

class PathLabelDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = list(paths)
        self.labels = list(labels)
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        y = self.labels[idx]
        img = default_loader(p)
        if img.mode != "RGB": img = img.convert("RGB")
        return img, y, p

SUPPORTED_EXTS = {".jpg",".jpeg",".png",".ppm",".bmp",".pgm",".tif",".tiff",".webp"}

def _norm_name(s: str):
    return s.lower().replace(" ", "").replace("-", "").replace("_", "")

def build_external_test2_dataset(root, train_class_to_idx):
    """
    Robust: scans recursively under TEST2_ROOT.
    Assigns label by nearest ancestor folder name matching a training class.
    """
    rootp = Path(root)
    if not rootp.exists():
        raise FileNotFoundError(f"TEST2_ROOT does not exist: {root}")

    norm_map = {_norm_name(k): v for k,v in train_class_to_idx.items()}
    img_paths, img_labels = [], []

    all_files = [p for p in rootp.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_EXTS]
    all_files = sorted(all_files, key=lambda x: str(x).lower())

    for p in all_files:
        lbl = None
        for parent in p.parents:
            nm = _norm_name(parent.name)
            if nm in norm_map:
                lbl = norm_map[nm]
                break
        if lbl is not None:
            img_paths.append(str(p))
            img_labels.append(int(lbl))

    if len(img_paths) == 0:
        subdirs = [d.name for d in rootp.iterdir() if d.is_dir()]
        raise FileNotFoundError(
            "TEST2 scan found 0 labeled images.\n"
            f"- Root: {root}\n"
            f"- Immediate subfolders: {subdirs[:30]}\n"
            f"- Expected class folders (somewhere in path): {list(train_class_to_idx.keys())}\n"
            f"- Supported extensions: {sorted(SUPPORTED_EXTS)}"
        )

    inv = {v:k for k,v in train_class_to_idx.items()}
    cnt = Counter(img_labels)
    print("\n" + "-"*120)
    print(f"EXTERNAL TEST2 (eval-only): {TEST2_NAME}")
    print(f"ROOT: {root}")
    print("Found images:", len(img_paths))
    print("Per-class:", {inv[k]: int(cnt.get(k,0)) for k in sorted(inv.keys())})
    print("-"*120)

    return PathLabelDataset(img_paths, img_labels)

def labels_for_folder(split_dir):
    ds = RawImageFolder(split_dir)
    return torch.tensor([lab for _,lab in ds.samples], dtype=torch.long)

# ============================= Backbones ======================================================
def pick_effnet_fallback():
    avail = set(timm.list_models("*efficientnet*"))
    for c in ["tf_efficientnetv2_m_in21ft1k","tf_efficientnetv2_s_in21ft1k","efficientnetv2_rw_s","tf_efficientnet_b3_ns"]:
        if c in avail: return c
    return None

def pick_dinov3():
    avail = set(timm.list_models("*dinov3*"))
    for c in ["vit_base_patch16_dinov3.lvd1689m","vit_base_patch16_dinov3","vit_base_patch16_dinov3.lvd142m"]:
        if c in avail: return c
    for m in sorted(avail):
        if "vit_base" in m and "patch16" in m:
            return m
    return None

class TimmBackbone(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.model_name = name
        is_dino = ("dinov2" in name) or ("dinov3" in name)
        if is_dino:
            self.model = timm.create_model(name, pretrained=True, num_classes=0)
        else:
            self.model = timm.create_model(name, pretrained=True, num_classes=0, global_pool="avg")

        self.model.eval().to(device)
        for p in self.model.parameters():
            p.requires_grad = False

        cfg = resolve_data_config(getattr(self.model, "pretrained_cfg", {}), model=self.model)
        self.transform = create_transform(**cfg, is_training=False)
        self.size = cfg.get("input_size",(3,224,224))[-2:]

        with torch.no_grad():
            d = torch.zeros(1,3,*self.size, device=device)
            out = self.forward(d)
            self.out_dim = int(out.shape[-1])

    @torch.no_grad()
    def forward(self, x):
        out = self.model(x)
        if out.ndim == 2: return out
        if out.ndim == 3: return out.mean(1)
        return out.view(out.size(0), -1)

# ============================= Fingerprint ====================================================
from skimage.filters import threshold_otsu
from skimage.exposure import histogram
from scipy.stats import skew, kurtosis

def modality_fingerprint(pil_img):
    g = np.array(pil_img.convert("L"), dtype=np.uint8)
    hist,_ = histogram(g, nbins=64, source_range="dtype")
    h = hist.astype(np.float32); h /= (h.sum()+1e-8)
    g_f = g.astype(np.float32)/255.0
    mu, sd = float(g_f.mean()), float(g_f.std()+1e-8)
    sk = float(np.nan_to_num(skew(g_f.reshape(-1), bias=False), nan=0.0))
    ku = float(np.nan_to_num(kurtosis(g_f.reshape(-1), fisher=True, bias=False), nan=0.0))
    ent = float(-(h*np.log(h+1e-8)).sum())
    try:
        t = threshold_otsu(g); fg_ratio = float((g>=t).mean())
    except Exception:
        fg_ratio = 0.5
    return np.concatenate([h, np.array([mu,sd,sk,ku,ent,fg_ratio], np.float32)])

def five_crops(img, size_hw):
    th, tw = size_hw
    w,h = img.size
    if h < th or w < tw:
        img = img.resize((max(w,tw), max(h,th)), resample=RESAMPLE_BILINEAR); w,h = img.size
    return [
        img.crop((0,0,tw,th)),
        img.crop((w-tw,0,w,th)),
        img.crop((0,h-th,tw,h)),
        img.crop((w-tw,h-th,w,h)),
        img.crop(((w-tw)//2,(h-th)//2,(w+tw)//2,(h+th)//2)),
    ]

# ============================= Feature caching ================================================
def _cache_fp_for_dataset(ds, tag, cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    fp_path = os.path.join(cache_dir, f"{tag}_fp.pt")
    if os.path.exists(fp_path):
        return torch.load(fp_path, map_location="cpu")

    loader = DataLoader(ds, batch_size=EXTRACT_BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                        collate_fn=lambda b: [x[0] for x in b])
    fps=[]
    for batch_imgs in tqdm(loader, desc=f"[FP] {tag}", ncols=100):
        fps.append(torch.from_numpy(np.stack([modality_fingerprint(im) for im in batch_imgs],0)).float())
    FP = torch.cat(fps,0)
    torch.save(FP, fp_path)
    return FP

@torch.no_grad()
def _extract_backbone_feats(ds, bb: TimmBackbone, tta_mode: str, tag: str, cache_dir: str):
    os.makedirs(cache_dir, exist_ok=True)
    bb_tag = bb.model_name.replace("/","-")
    fpath = os.path.join(cache_dir, f"{tag}_full_{bb_tag}_tta{tta_mode}.pt")
    if os.path.exists(fpath):
        return torch.load(fpath, map_location="cpu")

    loader = DataLoader(ds, batch_size=EXTRACT_BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                        collate_fn=lambda b: list(zip(*b)))

    feats=[]
    for batch in tqdm(loader, desc=f"[Extract {tag}|{bb_tag}|tta={tta_mode}]", ncols=100):
        imgs, ys, paths = batch
        x0 = torch.stack([bb.transform(im) for im in imgs]).to(device)
        f0 = bb(x0)

        if tag != "train" and tta_mode in ["flip","five"]:
            if tta_mode=="flip" and ALLOW_FLIP:
                xflip = torch.flip(x0, dims=[3])
                f1 = bb(xflip)
                f0 = 0.5*(f0+f1)
            elif tta_mode=="five":
                crops=[]
                for im in imgs:
                    for c in five_crops(im, bb.size):
                        crops.append(bb.transform(c))
                x5 = torch.stack(crops).to(device)
                f5 = bb(x5).view(len(imgs),5,-1).mean(1)
                f0 = 0.5*(f0 + f5)

        feats.append(f0.cpu())

    F = torch.cat(feats,0)
    if SAVE_CACHE_FP16:
        F = F.half()
    torch.save(F, fpath)
    return F

def cache_features_for_dataset(ds, tag, tta_mode, backbones, cache_dir):
    out = {}
    for bb in backbones:
        out[(bb.model_name.replace("/","-"), "full")] = _extract_backbone_feats(ds, bb, tta_mode, tag, cache_dir)
    if USE_FINGERPRINT:
        out[("fp","vec")] = _cache_fp_for_dataset(ds, tag, cache_dir)
    return out

def build_streams(cache_dict, backbones):
    def _as_float(t):
        if not isinstance(t, torch.Tensor): return t
        return t.float() if LOAD_CAST_FLOAT32 else t
    mats = [_as_float(cache_dict[(bb.model_name.replace("/","-"),"full")]) for bb in backbones]
    X_full = torch.cat(mats, 1)
    fp = cache_dict.get(("fp","vec"), None)
    if isinstance(fp, torch.Tensor): fp = fp.float()
    return X_full, fp

def fit_zscore(X):
    mu = X.mean(0, keepdim=True)
    sd = X.std(0, keepdim=True).clamp_min(1e-6)
    return mu, sd

def apply_zscore(X, mu, sd):
    return (X - mu)/sd

# ============================= Heads ==========================================================
def get_activation(n):
    return {"mish":nn.Mish(),"silu":nn.SiLU(),"gelu":nn.GELU(),"leakyrelu":nn.LeakyReLU(0.1)}[n]

class MLPHeadConcat(nn.Module):
    def __init__(self, in_dim, num_classes, activation="mish"):
        super().__init__()
        act = get_activation(activation)
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024), act, nn.Dropout(0.30),
            nn.Linear(1024, 256), act, nn.Dropout(0.20),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.net(x)

class AttentionFusionHead(nn.Module):
    def __init__(self, in_dims, fuse_dim, num_classes, fp_dim=0, activation="mish"):
        super().__init__()
        self.in_dims = list(map(int, in_dims))
        self.proj = nn.ModuleList([nn.Linear(d, fuse_dim) for d in self.in_dims])
        self.gate = nn.Sequential(nn.Linear(fuse_dim, fuse_dim//2), nn.GELU(), nn.Linear(fuse_dim//2, 1))
        self.fp_proj = nn.Linear(fp_dim, fuse_dim) if (fp_dim and fp_dim>0) else None
        act = get_activation(activation)
        fin = fuse_dim + (fuse_dim if self.fp_proj is not None else 0)
        self.clf = nn.Sequential(
            nn.Linear(fin, 1024), act, nn.Dropout(0.30),
            nn.Linear(1024, 256), act, nn.Dropout(0.20),
            nn.Linear(256, num_classes)
        )

    def forward(self, X, fp=None):
        expected = sum(self.in_dims)
        if X.size(1) != expected:
            if X.size(1) > expected: X = X[:, :expected]
            else: X = torch.cat([X, X.new_zeros(X.size(0), expected - X.size(1))], dim=1)

        parts, off = [], 0
        for d in self.in_dims:
            parts.append(X[:, off:off+d]); off += d
        Z = [proj(p) for proj,p in zip(self.proj, parts)]
        gates = [self.gate(z) for z in Z]
        A = torch.softmax(torch.stack(gates, dim=1).squeeze(-1), dim=1)
        fused = torch.sum(torch.stack(Z, dim=1) * A.unsqueeze(-1), dim=1)
        if self.fp_proj is not None and fp is not None:
            fused = torch.cat([fused, self.fp_proj(fp)], dim=1)
        return self.clf(fused)

class GatedFusionHead(nn.Module):
    def __init__(self, dims, hidden=2048, num_classes=4, p_drop=0.10):
        super().__init__()
        self.dims = list(map(int,dims))
        self.streams = len(self.dims)
        self.in_dim = sum(self.dims)
        self.gates = nn.Sequential(nn.Linear(self.in_dim, self.streams), nn.Sigmoid())
        self.norm  = nn.LayerNorm(self.in_dim)
        self.mlp   = nn.Sequential(nn.Linear(self.in_dim, hidden), nn.GELU(), nn.Dropout(p_drop), nn.Linear(hidden, num_classes))
    def forward(self, x):
        g = self.gates(x)
        out_parts=[]; off=0
        for i,d in enumerate(self.dims):
            out_parts.append(x[:,off:off+d] * g[:,i:i+1]); off += d
        xg = torch.cat(out_parts,1)
        return self.mlp(self.norm(xg))

# ============================= EMA + Efficiency helpers =======================================
class EMA:
    def __init__(self, module, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k:v.detach().clone() for k,v in module.state_dict().items()}
        self.module = module
        self.backup = None
    def update(self):
        with torch.no_grad():
            for k,v in self.module.state_dict().items():
                if k in self.shadow:
                    self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
                else:
                    self.shadow[k] = v.detach().clone()
    def apply(self):
        self.backup = {k:v.detach().clone() for k,v in self.module.state_dict().items()}
        self.module.load_state_dict(self.shadow, strict=False)
    def restore(self):
        if self.backup is not None:
            self.module.load_state_dict(self.backup, strict=False)
            self.backup = None

def count_params(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters()))

@torch.no_grad()
def measure_latency_ms_per_img(model, xb, fpb=None, warmup=40, repeats=120) -> float:
    model.eval().to(device)
    xb = xb.to(device, non_blocking=True)
    fpb = fpb.to(device, non_blocking=True) if fpb is not None else None

    for _ in range(warmup):
        _ = model(xb, fpb) if hasattr(model, "fp_proj") else model(xb)
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(repeats):
        _ = model(xb, fpb) if hasattr(model, "fp_proj") else model(xb)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    ms = (t1 - t0) * 1000.0 / (repeats * xb.size(0))
    return float(ms)

def prep_latency_batch(head_type, use_fp, X_full, fp, batch=256):
    b = min(batch, X_full.shape[0])
    if head_type in ("concat", "gated"):
        if use_fp and fp is not None:
            xb = torch.cat([X_full[:b], fp[:b]], dim=1).float()
        else:
            xb = X_full[:b].float()
        return xb, None
    xb = X_full[:b].float()
    fpb = fp[:b].float() if (use_fp and fp is not None) else None
    return xb, fpb

# ============================= Metrics =========================================================
def get_loss():
    try: return nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    except TypeError: return nn.CrossEntropyLoss()

@torch.no_grad()
def temperature_scale(logits, labels):
    T_candidates = np.linspace(0.3, 3.0, 55)
    bestT, bestNLL = 1.0, 1e9
    ce = nn.CrossEntropyLoss()
    for T in T_candidates:
        loss = ce(logits/float(T), labels).item()
        if loss < bestNLL:
            bestNLL, bestT = loss, float(T)
    return bestT

def compute_ece(probs, y_true, bins=10):
    probs = np.asarray(probs); y_true = np.asarray(y_true)
    probs = np.clip(probs, 0.0, 1.0)
    conf = probs.max(1); pred = probs.argmax(1); corr = (pred == y_true)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for i in range(bins):
        lo, hi = edges[i], edges[i+1]
        m = ((conf >= lo) & (conf < hi)) if i < bins-1 else ((conf >= lo) & (conf <= hi))
        if m.any():
            ece += abs(corr[m].mean() - conf[m].mean()) * m.mean()
    return float(ece)

def bootstrap_acc_ci(y_true, y_pred, B=1000, seed=123):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true); accs=[]
    for _ in range(B):
        idx = rng.integers(0, n, n)
        accs.append((y_pred[idx]==y_true[idx]).mean())
    accs = np.sort(accs)
    return float(np.mean(accs)), (float(np.percentile(accs,2.5)), float(np.percentile(accs,97.5)))

def brier_multi(y_true, probs):
    y_true = np.asarray(y_true); probs = np.asarray(probs)
    Y = label_binarize(y_true, classes=list(range(probs.shape[1])))
    return float(((probs - Y)**2).mean())

def detect_none_index(class_names):
    NONE_ALIASES = {"none","notumor","notumour","no_tumor","no-tumor","normal","healthy"}
    for i,c in enumerate(class_names):
        cc = _norm_name(c)
        if cc in {_norm_name(x) for x in NONE_ALIASES} or ("notumor" in cc) or (cc=="none"):
            return i
    return None

def detection_metrics_from_multiclass(y_true, probs, NONE_IDX, thresh):
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    p_none = probs[:, NONE_IDX]
    p_tumor = 1.0 - p_none
    y_det_true = (y_true != NONE_IDX).astype(int)
    y_det_pred = (p_tumor >= thresh).astype(int)
    det_acc = float((y_det_pred == y_det_true).mean())

    tumor_idxs = [i for i in range(probs.shape[1]) if i != NONE_IDX]
    tumor_map = {orig:i for i,orig in enumerate(tumor_idxs)}

    tumor_mask = (y_det_true == 1)
    if tumor_mask.any():
        y_tum_true = np.array([tumor_map[int(t)] for t in y_true[tumor_mask]], dtype=int)
        y_tum_pred = probs[tumor_mask][:, tumor_idxs].argmax(1)
        tumor_subtype_acc = float((y_tum_pred == y_tum_true).mean())
    else:
        tumor_subtype_acc = float("nan")

    tumor_pred_global = np.array(tumor_idxs, dtype=int)[probs[:, tumor_idxs].argmax(1)]
    y_pred_pipeline = np.where(y_det_pred==0, NONE_IDX, tumor_pred_global)
    pipeline_acc = float((y_pred_pipeline == y_true).mean())

    return det_acc, tumor_subtype_acc, pipeline_acc

def safe_macro_metrics(y_true, probs):
    y_true = np.asarray(y_true); probs = np.asarray(probs)
    K = probs.shape[1]
    Y = label_binarize(y_true, classes=list(range(K)))
    try:
        auroc = float(roc_auc_score(Y, probs, average="macro", multi_class="ovr"))
    except Exception:
        auroc = float("nan")
    try:
        ap = float(average_precision_score(Y, probs, average="macro"))
    except Exception:
        ap = float("nan")
    return auroc, ap

def compute_all_metrics(y_true, probs, NONE_IDX, thresh, seed_for_ci=SEED):
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    preds = probs.argmax(1)

    cls_acc = float((preds == y_true).mean())
    macroF1 = float(f1_score(y_true, preds, average="macro"))
    macroAUROC, macroAP = safe_macro_metrics(y_true, probs)
    ece = compute_ece(probs, y_true, bins=10)
    brier = brier_multi(y_true, probs)
    _, (lo, hi) = bootstrap_acc_ci(y_true, preds, B=1000, seed=seed_for_ci)

    det_acc, tumor_sub_acc, pipeline_acc = detection_metrics_from_multiclass(y_true, probs, NONE_IDX, thresh)
    return dict(
        cls_acc=cls_acc, cls_acc_CI=(lo,hi),
        macroF1=macroF1, macroAUROC=macroAUROC, macroAP=macroAP,
        ECE=ece, Brier=brier,
        det_acc=det_acc, tumor_subtype_acc=tumor_sub_acc, pipeline_acc=pipeline_acc
    )

# ============================= Train/Eval helpers ============================================
def X_size(X):
    return X[0].size(0) if isinstance(X,(list,tuple)) else X.size(0)

def X_gather(X, idx):
    if isinstance(X,(list,tuple)):
        parts = [t.index_select(0, idx) for t in X if t is not None]
        return torch.cat(parts, 1)
    return X.index_select(0, idx)

def X_slice(X, start, end):
    if isinstance(X,(list,tuple)):
        parts = [t[start:end] for t in X if t is not None]
        return torch.cat(parts, 1)
    return X[start:end]

def train_head(model, Xtr, ytr, Xva, yva, fp_tr=None, fp_va=None):
    t_start = time.perf_counter()

    model = model.to(device)
    opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = get_loss()
    num_train = X_size(Xtr)
    steps = EPOCHS * max(1, math.ceil(num_train / BATCH_SIZE))

    class WarmupCosine:
        def __init__(self, opt, total_steps, warmup_frac, base_lr, min_lr=1e-6):
            self.opt=opt; self.total=total_steps; self.warm=int(total_steps*warmup_frac)
            self.base_lr=base_lr; self.min_lr=min_lr; self.i=0
        def step(self):
            self.i += 1
            if self.i <= self.warm:
                lr = self.base_lr * self.i / max(1, self.warm)
            else:
                t = (self.i - self.warm) / max(1, self.total - self.warm)
                lr = self.min_lr + 0.5*(self.base_lr - self.min_lr)*(1 + math.cos(math.pi * t))
            for pg in self.opt.param_groups: pg["lr"] = lr

    sched = WarmupCosine(opt, steps, WARMUP_FRAC, LR)
    scaler = GradScaler(enabled=USE_AMP and torch.cuda.is_available() and _amp_dtype == torch.float16)
    ema = EMA(model, EMA_DECAY) if USE_EMA else None

    best_acc, best_state, best_epoch, patience = 0.0, None, 0, 8

    import inspect as _inspect
    has_device_type = ("device_type" in _inspect.signature(autocast).parameters)

    for epoch in range(1, EPOCHS+1):
        model.train()
        perm = torch.randperm(num_train)
        for i in range(0, num_train, BATCH_SIZE):
            idx = perm[i:i+BATCH_SIZE]
            xb = X_gather(Xtr, idx).to(device, non_blocking=True)
            yb = ytr.index_select(0, idx).to(device, non_blocking=True)
            fpb = (fp_tr.index_select(0, idx).to(device, non_blocking=True) if fp_tr is not None else None)

            mix_lam = 1.0
            if MIXUP_ALPHA > 0 and xb.size(0)>1 and random.random()<0.5:
                mix_lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                perm2 = torch.randperm(yb.size(0), device=yb.device)
                xb = mix_lam*xb + (1.0-mix_lam)*xb[perm2]
                if fpb is not None:
                    fpb = mix_lam*fpb + (1.0-mix_lam)*fpb[perm2]

            opt.zero_grad(set_to_none=True)
            ac_kwargs = dict(enabled=USE_AMP and torch.cuda.is_available(), dtype=_amp_dtype)
            if has_device_type: ac_kwargs["device_type"] = "cuda"

            with autocast(**ac_kwargs):
                logits = model(xb, fpb) if hasattr(model,"fp_proj") else model(xb)
                if mix_lam != 1.0:
                    loss = mix_lam*F.cross_entropy(logits, yb) + (1.0-mix_lam)*F.cross_entropy(logits, yb[perm2])
                else:
                    loss = loss_fn(logits, yb)

            if USE_AMP and scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward(); opt.step()

            sched.step()
            if ema: ema.update()

        # validate
        model.eval()
        if ema: ema.apply()
        with torch.no_grad():
            val_logits=[]
            n_val = X_size(Xva)
            for k in range(0, n_val, 256):
                xb = X_slice(Xva, k, k+256).to(device, non_blocking=True)
                fpb = (fp_va[k:k+256].to(device, non_blocking=True) if fp_va is not None else None)
                val_logits.append((model(xb, fpb) if hasattr(model,"fp_proj") else model(xb)).cpu())
            logits_v = torch.cat(val_logits,0)
            val_acc = float((logits_v.argmax(1).numpy()==yva.numpy()).mean())
        if ema: ema.restore()

        if val_acc > best_acc:
            best_acc = val_acc; best_epoch = epoch; patience = 8
            if USE_EMA_FOR_SELECTION and ema:
                best_state = {k:v.detach().cpu().clone() for k,v in ema.shadow.items()}
            else:
                best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            patience -= 1
            if patience <= 0: break

    if isinstance(best_state, dict):
        model.load_state_dict(best_state, strict=False)

    t_end = time.perf_counter()
    return model, float(best_acc), int(best_epoch), float(t_end - t_start)

@torch.no_grad()
def run_logits(model, X, fp=None):
    model.eval()
    outs=[]
    n = X_size(X)
    for i in range(0, n, 256):
        xb = X_slice(X, i, i+256).to(device)
        fpb = (fp[i:i+256].to(device) if fp is not None else None)
        l = model(xb, fpb) if hasattr(model,"fp_proj") else model(xb)
        outs.append(l.cpu())
    return torch.cat(outs,0)

# ============================= Variant evaluator =============================================
def evaluate_variant(
    name, head_type, use_fp, mode,
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, class_names, NONE_IDX,
    out_dir
):
    set_all_seeds(SEED)
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n=== Variant: {name} | head={head_type} | FP={use_fp} | mode={mode} ===")

    dims_each = list(bb_dims)
    fp_tr_in = fp_va_in = fp_te_in = fp2_in = None

    def _pack_concat_or_gated(Xmain, FP):
        return (Xmain, FP) if (use_fp and FP is not None) else Xmain

    if head_type in ("concat","gated"):
        Xtr = _pack_concat_or_gated(Xtr_full, fp_tr)
        Xva = _pack_concat_or_gated(Xva_full, fp_va)
        Xte = _pack_concat_or_gated(Xte_full, fp_te)
        X2  = _pack_concat_or_gated(X2_full,  fp2)

        if head_type=="concat":
            in_dim = int(Xtr_full.shape[1] + (fp_tr.shape[1] if (use_fp and fp_tr is not None) else 0))
        else:
            if use_fp and fp_tr is not None:
                dims_each = dims_each + [int(fp_tr.shape[1])]
    else:  # attn
        Xtr, Xva, Xte, X2 = Xtr_full, Xva_full, Xte_full, X2_full
        if use_fp and fp_tr is not None:
            fp_tr_in, fp_va_in, fp_te_in, fp2_in = fp_tr, fp_va, fp_te, fp2

    def _make_model(act_name):
        if head_type=="concat":
            return MLPHeadConcat(in_dim, num_classes, activation=act_name)
        if head_type=="attn":
            fp_dim = int(fp_tr_in.shape[1]) if fp_tr_in is not None else 0
            return AttentionFusionHead(dims_each, fuse_dim=ATTN_FUSE_DIM, num_classes=num_classes, fp_dim=fp_dim, activation=act_name)
        return GatedFusionHead(dims_each, hidden=2048, num_classes=num_classes, p_drop=0.10)

    trained, val_accs, train_times = [], [], []
    params_each, lat_each = [], []

    act_list = [SINGLE_ACT] if mode=="single" else list(ENSEMBLE_ACTS)
    act_off = {"mish":1,"silu":2,"gelu":3,"leakyrelu":4}

    xb_lat, fpb_lat = prep_latency_batch(head_type, use_fp, Xva_full, fp_va, batch=256)

    for act in act_list:
        set_all_seeds(SEED*10 + act_off.get(act,0))
        mdl = _make_model(act)
        mdl, vacc, be, tsec = train_head(mdl, Xtr, ytr, Xva, yva, fp_tr_in, fp_va_in)
        trained.append(mdl); val_accs.append(float(vacc)); train_times.append(float(tsec))

        pcount = count_params(mdl)
        params_each.append(pcount)
        lat = measure_latency_ms_per_img(mdl, xb_lat, fpb_lat)
        lat_each.append(float(lat))

        torch.save(mdl.state_dict(), os.path.join(out_dir, f"{name}_{act}_best.pth"))
        print(f"  [train] act={act} -> best_epoch={be} val_acc={vacc:.4f} | train_s={tsec:.1f} | params={pcount/1e6:.2f}M | lat={lat:.6f} ms/img")

    # ensemble logits (metrics)
    if len(trained)==1:
        logits_va = run_logits(trained[0], Xva, fp_va_in)
        logits_te = run_logits(trained[0], Xte, fp_te_in)
        logits_2  = run_logits(trained[0], X2,  fp2_in) if X2_full is not None else None
    else:
        lv = torch.stack([run_logits(m, Xva, fp_va_in) for m in trained], 0)
        lt = torch.stack([run_logits(m, Xte, fp_te_in) for m in trained], 0)
        l2 = torch.stack([run_logits(m, X2,  fp2_in)  for m in trained], 0) if X2_full is not None else None

        w = torch.tensor(val_accs, dtype=lv.dtype)
        if float(w.max()-w.min()) < 1e-6:
            logits_va, logits_te = lv.mean(0), lt.mean(0)
            logits_2 = l2.mean(0) if l2 is not None else None
        else:
            w = (w - w.min()) / (w.max() - w.min() + 1e-6)
            w = 0.5 + 0.5*w
            W = w.view(-1,1,1)
            logits_va = (W*lv).sum(0) / W.sum()
            logits_te = (W*lt).sum(0) / W.sum()
            logits_2  = ((W*l2).sum(0) / W.sum()) if l2 is not None else None

    best_i = int(np.argmax(val_accs))  # keep for val_acc_best bookkeeping

# Store ENSEMBLE raw logits for stacking (single variants are just 1-model ensembles)
    best_val_logits_raw  = logits_va.cpu().numpy()
    best_test_logits_raw = logits_te.cpu().numpy()
    best_test2_logits_raw = logits_2.cpu().numpy() if logits_2 is not None else None
    T = 1.0
    if DO_TEMPERATURE_SCALING:
        T = temperature_scale(logits_va, yva)
        logits_va = logits_va/float(T)
        logits_te = logits_te/float(T)
        if logits_2 is not None: logits_2 = logits_2/float(T)

    probs_va = torch.softmax(logits_va,1).numpy()
    probs_te = torch.softmax(logits_te,1).numpy()
    probs_2  = torch.softmax(logits_2,1).numpy() if logits_2 is not None else None

    m_val  = compute_all_metrics(yva.numpy(), probs_va, NONE_IDX, DETECT_THRESH)
    m_test = compute_all_metrics(yte.numpy(), probs_te, NONE_IDX, DETECT_THRESH)
    m_2    = compute_all_metrics(y2.numpy(),  probs_2,  NONE_IDX, DETECT_THRESH) if (probs_2 is not None and y2 is not None) else None

    eff_train_s = float(sum(train_times))
    eff_params  = int(sum(params_each))
    eff_lat_ms  = float(sum(lat_each))

    return dict(
        name=name, head_type=head_type, use_fp=use_fp, mode=mode,
        T=float(T),
        val=m_val, test=m_test, test2=m_2,

        # per-sample probabilities for EXPORT/Bowker
        val_probs=probs_va,
        test_probs=probs_te,
        test2_probs=probs_2,

        # for stacking/super
        val_acc_best=float(val_accs[best_i]),
        best_val_logits_raw=best_val_logits_raw,
        best_test_logits_raw=best_test_logits_raw,
        best_test2_logits_raw=best_test2_logits_raw,

        # efficiency
        train_time_s=eff_train_s,
        params=eff_params,
        latency_ms=eff_lat_ms,
    )

# ============================= Super-Ensemble (best SINGLE per head_type) =====================
def evaluate_super_ensemble(base_records, yva, yte, y2, NONE_IDX, out_dir, name="super_bestheads"):
    chosen={}
    for r in base_records:
        if r.get("mode") != "single":
            continue
        h = r.get("head_type")
        if h is None: continue
        if h not in chosen or r["val_acc_best"] > chosen[h]["val_acc_best"]:
            chosen[h] = r
    picked = list(chosen.values())
    assert len(picked) >= 2, "Need >=2 head types for super-ensemble."

    Lva = [torch.from_numpy(r["best_val_logits_raw"])  for r in picked]
    Lte = [torch.from_numpy(r["best_test_logits_raw"]) for r in picked]
    L2  = [torch.from_numpy(r["best_test2_logits_raw"]) for r in picked if r["best_test2_logits_raw"] is not None]
    have_test2 = (y2 is not None) and (len(L2)==len(Lva))

    W = torch.tensor([r["val_acc_best"] for r in picked], dtype=torch.float32).view(-1,1,1)
    W = (W - W.min()) / (W.max() - W.min() + 1e-6); W = 0.5 + 0.5*W

    logits_va = (torch.stack(Lva,0) * W).sum(0) / W.sum()
    logits_te = (torch.stack(Lte,0) * W).sum(0) / W.sum()
    logits_2  = ((torch.stack(L2,0)  * W).sum(0) / W.sum()) if have_test2 else None

    T=1.0
    if DO_TEMPERATURE_SCALING:
        T = temperature_scale(logits_va, yva)
        logits_va = logits_va/float(T)
        logits_te = logits_te/float(T)
        if logits_2 is not None: logits_2 = logits_2/float(T)

    probs_va = torch.softmax(logits_va,1).numpy()
    probs_te = torch.softmax(logits_te,1).numpy()
    probs_2  = torch.softmax(logits_2,1).numpy() if logits_2 is not None else None

    eff_train_s = float(sum(r.get("train_time_s",0.0) for r in picked))
    eff_params  = int(sum(r.get("params",0) for r in picked))
    eff_lat_ms  = float(sum(r.get("latency_ms",0.0) for r in picked))

    return dict(
        name=name, T=float(T),
        val=compute_all_metrics(yva.numpy(), probs_va, NONE_IDX, DETECT_THRESH),
        test=compute_all_metrics(yte.numpy(), probs_te, NONE_IDX, DETECT_THRESH),
        test2=(compute_all_metrics(y2.numpy(), probs_2, NONE_IDX, DETECT_THRESH) if (probs_2 is not None and y2 is not None) else None),

        val_probs=probs_va,
        test_probs=probs_te,
        test2_probs=probs_2,

        train_time_s=eff_train_s,
        params=eff_params,
        latency_ms=eff_lat_ms,
    )

# ============================= Stacking (VAL-driven) ==========================================
def evaluate_stacking_val_driven(results, yva, yte, y2, NONE_IDX, name, base_variants, kfold=5):
    cost = {r["name"]: r for r in results if isinstance(r, dict) and "name" in r}

    selected=[]
    want=set(base_variants)
    for r in results:
        if r.get("name") in want and "best_val_logits_raw" in r:
            selected.append(r)
    assert len(selected) >= 2, f"{name}: need >=2 base variants"

    t0 = time.perf_counter()

    val_probs_list=[]; test_probs_list=[]; test2_probs_list=[]
    for r in selected:
        Lv = torch.from_numpy(r["best_val_logits_raw"])
        Lt = torch.from_numpy(r["best_test_logits_raw"])
        L2 = torch.from_numpy(r["best_test2_logits_raw"]) if (r.get("best_test2_logits_raw") is not None) else None

        Ti = temperature_scale(Lv, yva)
        val_probs_list.append(torch.softmax(Lv/float(Ti),1).numpy())
        test_probs_list.append(torch.softmax(Lt/float(Ti),1).numpy())
        if (y2 is not None) and (L2 is not None):
            test2_probs_list.append(torch.softmax(L2/float(Ti),1).numpy())

    Xv = np.concatenate(val_probs_list, axis=1)
    Xt = np.concatenate(test_probs_list, axis=1)
    X2 = np.concatenate(test2_probs_list, axis=1) if ((y2 is not None) and (len(test2_probs_list)==len(val_probs_list))) else None

    yv = yva.numpy(); yt = yte.numpy(); y2n = y2.numpy() if y2 is not None else None
    K = val_probs_list[0].shape[1]

    skf = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=SEED)
    oof = np.zeros((len(yv), K), dtype=float)

    for tr_idx, va_idx in skf.split(Xv, yv):
        scaler = StandardScaler(with_mean=False)
        Xtr_s = scaler.fit_transform(Xv[tr_idx])
        Xva_s = scaler.transform(Xv[va_idx])
        meta = LogisticRegression(penalty="l2", C=1.0, max_iter=2000, multi_class="multinomial", solver="lbfgs")
        meta.fit(Xtr_s, yv[tr_idx])
        oof[va_idx] = meta.predict_proba(Xva_s)

    scaler = StandardScaler(with_mean=False)
    Xv_s = scaler.fit_transform(Xv)
    Xt_s = scaler.transform(Xt)
    meta = LogisticRegression(penalty="l2", C=1.0, max_iter=2000, multi_class="multinomial", solver="lbfgs")
    meta.fit(Xv_s, yv)
    Pt = meta.predict_proba(Xt_s)

    P2 = None
    if X2 is not None:
        X2_s = scaler.transform(X2)
        P2 = meta.predict_proba(X2_s)

    meta_train_s = time.perf_counter() - t0

    base_train = sum(cost[n].get("train_time_s", 0.0) for n in base_variants if n in cost)
    base_params = sum(cost[n].get("params", 0) for n in base_variants if n in cost)
    base_lat = sum(cost[n].get("latency_ms", 0.0) for n in base_variants if n in cost)

    H = len(base_variants)
    meta_params = int(K*(K*H) + K)  # 36 for top2, 52 for top3 when K=4
    eff_train_s = float(base_train + meta_train_s)
    eff_params  = int(base_params + meta_params)
    eff_lat_ms  = float(base_lat)

    return dict(
        name=name, T=1.0,
        val=compute_all_metrics(yv, oof, NONE_IDX, DETECT_THRESH),
        test=compute_all_metrics(yt, Pt,  NONE_IDX, DETECT_THRESH),
        test2=(compute_all_metrics(y2n, P2, NONE_IDX, DETECT_THRESH) if (P2 is not None and y2 is not None) else None),

        val_probs=oof,
        test_probs=Pt,
        test2_probs=P2,

        train_time_s=eff_train_s,
        params=eff_params,
        latency_ms=eff_lat_ms,
        meta_train_time_s=float(meta_train_s),
        meta_params=meta_params
    )

# ============================= TRAIN-only OOF stacking (leakage-free) =========================
def evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs, kfold=5, inner_val_frac=0.10,
    name="stack_oof", within_head_ensemble=False,
    base_cost_lookup=None
):
    base_cost_lookup = base_cost_lookup or {}
    set_all_seeds(SEED)

    N_tr = Xtr_full.shape[0]
    N_va = Xva_full.shape[0]
    N_te = Xte_full.shape[0]
    N_2  = X2_full.shape[0] if X2_full is not None else 0

    def _build_head(head_type, use_fp, activation):
        if head_type == "concat":
            in_dim = int(Xtr_full.shape[1]) + (int(fp_tr.shape[1]) if (use_fp and fp_tr is not None) else 0)
            return MLPHeadConcat(in_dim, num_classes, activation=activation)
        if head_type == "attn":
            fp_dim = int(fp_tr.shape[1]) if (use_fp and fp_tr is not None) else 0
            return AttentionFusionHead(bb_dims, fuse_dim=ATTN_FUSE_DIM, num_classes=num_classes, fp_dim=fp_dim, activation=activation)
        dims = list(bb_dims)
        if use_fp and fp_tr is not None:
            dims = dims + [int(fp_tr.shape[1])]
        return GatedFusionHead(dims, hidden=2048, num_classes=num_classes, p_drop=0.10)

    def _pack(head_type, use_fp, Xmain, FP):
        if head_type in ("concat","gated") and use_fp and FP is not None:
            return (Xmain, FP), None
        if head_type=="attn" and use_fp and FP is not None:
            return Xmain, FP
        return Xmain, None

    def _select_rows(Xmain, idx):
        idx_t = torch.as_tensor(idx, dtype=torch.long)
        if isinstance(Xmain, tuple):
            return (Xmain[0].index_select(0, idx_t), Xmain[1].index_select(0, idx_t))
        return Xmain.index_select(0, idx_t)

    per_head={}
    for hs in head_specs:
        per_head[hs] = dict(
            oof=np.zeros((N_tr, num_classes), dtype=np.float32),
            val_sum=np.zeros((N_va, num_classes), dtype=np.float32),
            te_sum=np.zeros((N_te, num_classes), dtype=np.float32),
            t2_sum=np.zeros((N_2,  num_classes), dtype=np.float32) if X2_full is not None else None
        )

    skf = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=SEED)
    folds = list(skf.split(np.zeros(N_tr), ytr.numpy()))
    print(f"\n[STACK-OOF] {name} | folds={kfold} | within_head_ens={within_head_ensemble}")

    train_time_heads = 0.0

    for f,(tr_idx,oof_idx) in enumerate(folds,1):
        print(f"  Fold {f}/{kfold} | train={len(tr_idx)} | oof={len(oof_idx)}")
        sss = StratifiedShuffleSplit(n_splits=1, test_size=inner_val_frac, random_state=SEED+f)
        tr_sub_idx, inner_idx = next(sss.split(tr_idx, ytr.numpy()[tr_idx]))
        tr_sub = tr_idx[tr_sub_idx]
        inner  = tr_idx[inner_idx]

        y_trsub = torch.from_numpy(ytr.numpy()[tr_sub])
        y_inner = torch.from_numpy(ytr.numpy()[inner])

        for (head_type,use_fp) in head_specs:
            Xtr_pack, _ = _pack(head_type, use_fp, Xtr_full, fp_tr)
            Xva_pack, FPva_attn = _pack(head_type, use_fp, Xva_full, fp_va)
            Xte_pack, FPte_attn = _pack(head_type, use_fp, Xte_full, fp_te)
            X2_pack,  FP2_attn  = _pack(head_type, use_fp, X2_full,  fp2) if X2_full is not None else (None,None)

            X_trsub = _select_rows(Xtr_pack, tr_sub)
            X_inner = _select_rows(Xtr_pack, inner)
            X_oof   = _select_rows(Xtr_pack, oof_idx)

            fp_trsub = (fp_tr.index_select(0, torch.as_tensor(tr_sub)) if (head_type=="attn" and use_fp and fp_tr is not None) else None)
            fp_inner = (fp_tr.index_select(0, torch.as_tensor(inner))  if (head_type=="attn" and use_fp and fp_tr is not None) else None)
            fp_oof   = (fp_tr.index_select(0, torch.as_tensor(oof_idx)) if (head_type=="attn" and use_fp and fp_tr is not None) else None)

            accs=[]
            L_inner_list=[]; L_oof_list=[]; L_val_list=[]; L_te_list=[]; L_2_list=[]

            for act in ENSEMBLE_ACTS:
                set_all_seeds(SEED*100 + f*10 + {"mish":1,"silu":2,"gelu":3}[act])
                mdl = _build_head(head_type, use_fp, act)

                mdl, vacc, _, tsec = train_head(mdl, X_trsub, y_trsub, X_inner, y_inner, fp_trsub, fp_inner)
                train_time_heads += float(tsec)
                accs.append(float(vacc))

                def _rl(model, Xin, Finn):
                    return run_logits(model, Xin, fp=Finn) if head_type=="attn" else run_logits(model, Xin, fp=None)

                L_inner_list.append(_rl(mdl, X_inner, fp_inner))
                L_oof_list.append(  _rl(mdl, X_oof,   fp_oof))
                L_val_list.append(  _rl(mdl, Xva_pack, FPva_attn))
                L_te_list.append(   _rl(mdl, Xte_pack, FPte_attn))
                if X2_full is not None and y2 is not None:
                    L_2_list.append(_rl(mdl, X2_pack, FP2_attn))

            if not within_head_ensemble:
                bi = int(np.argmax(accs))
                L_inner = L_inner_list[bi]; L_oof = L_oof_list[bi]; L_val=L_val_list[bi]; L_te=L_te_list[bi]
                L_2 = L_2_list[bi] if (X2_full is not None and y2 is not None) else None
            else:
                w = np.array(accs, dtype=np.float32)
                if w.max()-w.min() < 1e-6:
                    W = None
                else:
                    w = (w - w.min())/(w.max()-w.min()+1e-6); w = 0.5+0.5*w
                    W = torch.from_numpy(w.reshape(-1,1,1))
                def _wavg(Ls):
                    if W is None: return torch.stack(Ls,0).mean(0)
                    return (torch.stack(Ls,0)*W).sum(0)/W.sum()
                L_inner = _wavg(L_inner_list)
                L_oof   = _wavg(L_oof_list)
                L_val   = _wavg(L_val_list)
                L_te    = _wavg(L_te_list)
                L_2     = _wavg(L_2_list) if (X2_full is not None and y2 is not None) else None

            T_head = temperature_scale(L_inner, y_inner)
            P_oof = torch.softmax(L_oof/float(T_head),1).numpy()
            P_val = torch.softmax(L_val/float(T_head),1).numpy()
            P_te  = torch.softmax(L_te/float(T_head),1).numpy()
            P_2   = torch.softmax(L_2/float(T_head),1).numpy() if L_2 is not None else None

            per_head[(head_type,use_fp)]["oof"][oof_idx] += P_oof
            per_head[(head_type,use_fp)]["val_sum"]      += P_val
            per_head[(head_type,use_fp)]["te_sum"]       += P_te
            if X2_full is not None and y2 is not None:
                per_head[(head_type,use_fp)]["t2_sum"]  += P_2

    # avg folds
    for k in per_head.keys():
        per_head[k]["val"]  = per_head[k]["val_sum"] / float(kfold)
        per_head[k]["test"] = per_head[k]["te_sum"]  / float(kfold)
        if X2_full is not None and y2 is not None:
            per_head[k]["test2"] = per_head[k]["t2_sum"] / float(kfold)
        del per_head[k]["val_sum"], per_head[k]["te_sum"]
        if X2_full is not None and y2 is not None:
            del per_head[k]["t2_sum"]

    head_keys = list(per_head.keys())
    X_oof  = np.concatenate([per_head[k]["oof"]  for k in head_keys], axis=1)
    X_val  = np.concatenate([per_head[k]["val"]  for k in head_keys], axis=1)
    X_test = np.concatenate([per_head[k]["test"] for k in head_keys], axis=1)
    X_t2   = np.concatenate([per_head[k]["test2"] for k in head_keys], axis=1) if (X2_full is not None and y2 is not None) else None

    y_train = ytr.numpy(); y_val = yva.numpy(); y_test = yte.numpy()
    y_t2 = y2.numpy() if y2 is not None else None

    # META OOF + final fit
    t_meta0 = time.perf_counter()
    skf2 = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=SEED)
    oof_meta = np.zeros((N_tr, num_classes), dtype=float)
    for tr_idx, oof_idx in skf2.split(X_oof, y_train):
        scaler = StandardScaler(with_mean=False)
        Xtr_s = scaler.fit_transform(X_oof[tr_idx])
        Xva_s = scaler.transform(X_oof[oof_idx])
        meta = LogisticRegression(penalty="l2", C=1.0, max_iter=2000, multi_class="multinomial", solver="lbfgs")
        meta.fit(Xtr_s, y_train[tr_idx])
        oof_meta[oof_idx] = meta.predict_proba(Xva_s)

    scaler = StandardScaler(with_mean=False)
    Xtr_s = scaler.fit_transform(X_oof)
    Xv_s  = scaler.transform(X_val)
    Xt_s  = scaler.transform(X_test)
    meta = LogisticRegression(penalty="l2", C=1.0, max_iter=2000, multi_class="multinomial", solver="lbfgs")
    meta.fit(Xtr_s, y_train)
    Pv = meta.predict_proba(Xv_s)
    Pt = meta.predict_proba(Xt_s)
    P2 = meta.predict_proba(scaler.transform(X_t2)) if X_t2 is not None else None
    meta_train_s = float(time.perf_counter() - t_meta0)

    # fair inference cost: sum constituent heads (single vs ens)
    inf_names = []
    for (ht, uf) in head_specs:
        inf_names.append(f"{ht}_{'fp' if uf else 'nofp'}_{'ens' if within_head_ensemble else 'single'}")

    base_params = sum(base_cost_lookup.get(n, {}).get("params", 0) for n in inf_names)
    base_lat    = sum(base_cost_lookup.get(n, {}).get("latency_ms", 0.0) for n in inf_names)

    K = num_classes
    H = len(inf_names)
    meta_params = int(K*(K*H) + K)
    eff_params = int(base_params + meta_params)
    eff_lat_ms = float(base_lat)

    total_train_s = float(train_time_heads + meta_train_s)

    return dict(
        name=name, T=1.0,
        val=compute_all_metrics(y_val, Pv, NONE_IDX, DETECT_THRESH),
        test=compute_all_metrics(y_test,Pt, NONE_IDX, DETECT_THRESH),
        test2=(compute_all_metrics(y_t2, P2, NONE_IDX, DETECT_THRESH) if (P2 is not None and y2 is not None) else None),

        val_probs=Pv,
        test_probs=Pt,
        test2_probs=P2,

        train_time_s=total_train_s,
        params=eff_params,
        latency_ms=eff_lat_ms,
        meta_train_time_s=meta_train_s,
        meta_params=meta_params,
        oof_heads_train_time_s=float(train_time_heads),
    )

# ============================= EXPORT + Efficiency + Radar ====================================
def export_predictions_for_all(results, OUT_DIR, ds_test, yte, class_names, ds_test2=None, y2=None):
    def _write_pred_csv(out_path, paths, y_true, probs, class_names):
        preds = probs.argmax(1)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        with open(out_path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["path","y_true","y_pred"] + [f"p_{c}" for c in class_names])
            for p, yt, yp, pr in zip(paths, y_true, preds, probs):
                w.writerow([p, int(yt), int(yp)] + [float(x) for x in pr])

    def _write_cm_csv(out_path, y_true, y_pred, K):
        cm = np.zeros((K, K), dtype=int)
        for yt, yp in zip(y_true, y_pred):
            cm[int(yt), int(yp)] += 1
        np.savetxt(out_path, cm, delimiter=",", fmt="%d")

    test_paths = [p for (p, _) in ds_test.samples]
    test_ytrue = np.asarray(yte.numpy(), dtype=int)
    K = len(class_names)

    test2_paths = test2_ytrue = None
    if ds_test2 is not None and y2 is not None:
        test2_paths = list(ds_test2.paths)
        test2_ytrue = np.asarray(y2.numpy(), dtype=int)

    exported, missing = [], []
    for r in results:
        name = r["name"]
        P = r.get("test_probs", None)
        if P is None:
            missing.append(name); continue
        P = np.asarray(P, dtype=np.float64)

        out_csv = os.path.join(OUT_DIR, f"{name}_test_predictions.csv")
        _write_pred_csv(out_csv, test_paths, test_ytrue, P, class_names)
        _write_cm_csv(os.path.join(OUT_DIR, f"{name}_test_cm.csv"), test_ytrue, P.argmax(1), K)

        if test2_paths is not None:
            P2 = r.get("test2_probs", None)
            if P2 is not None:
                P2 = np.asarray(P2, dtype=np.float64)
                out_csv2 = os.path.join(OUT_DIR, f"{name}_test2_predictions.csv")
                _write_pred_csv(out_csv2, test2_paths, test2_ytrue, P2, class_names)

        exported.append(name)

    print(f"\n[EXPORT] Wrote *_test_predictions.csv for: {len(exported)} / {len(results)}")
    if missing:
        print("[EXPORT] Missing test_probs for:", len(missing), "| examples:", missing[:10])

def make_efficiency_tables_and_radar(results, OUT_DIR):
    rows=[]
    for r in results:
        if "train_time_s" not in r or "params" not in r or "latency_ms" not in r:
            continue
        train_s = float(r.get("train_time_s", np.nan))
        params  = float(r.get("params", np.nan))
        lat_ms  = float(r.get("latency_ms", np.nan))
        if not np.isfinite(lat_ms) or lat_ms <= 0:
            lat_ms = np.nan
        speed = (1.0/lat_ms) if np.isfinite(lat_ms) else np.nan  # higher better
        rows.append([r["name"], train_s, params/1e6, lat_ms, speed])

    rows = sorted(rows, key=lambda x: (np.nan_to_num(x[1], nan=1e18), np.nan_to_num(x[2], nan=1e18)))
    arr = np.array(rows, dtype=object)
    names = arr[:,0].tolist()
    train_s = arr[:,1].astype(float)
    params_M = arr[:,2].astype(float)
    lat_ms = arr[:,3].astype(float)
    speed = arr[:,4].astype(float)

    def norm_inv(x):
        mn, mx = np.nanmin(x), np.nanmax(x)
        if mx - mn < 1e-12: return np.ones_like(x)
        return (mx - x) / (mx - mn)
    def norm_pos(x):
        mn, mx = np.nanmin(x), np.nanmax(x)
        if mx - mn < 1e-12: return np.ones_like(x)
        return (x - mn) / (mx - mn)

    score_train = norm_inv(train_s)
    score_params= norm_inv(params_M)
    score_speed = norm_pos(speed)
    efficiency  = (score_train + score_params + score_speed) / 3.0

    order = np.argsort(-efficiency)  # descending
    out_csv = os.path.join(OUT_DIR, "efficiency_summary.csv")
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["rank_efficiency","name","efficiency","train_time_s","params_M","latency_ms_per_img","score_train","score_params","score_speed"])
        for rk, i in enumerate(order, start=1):
            w.writerow([rk, names[i], float(efficiency[i]), float(train_s[i]), float(params_M[i]), float(lat_ms[i]),
                        float(score_train[i]), float(score_params[i]), float(score_speed[i])])

    print("\n=== Efficiency Summary (backbone excluded; higher efficiency better) ===")
    for rk, i in enumerate(order[:15], start=1):
        print(f"{rk:2d}. {names[i]:30s} | eff={efficiency[i]:.3f} | train={train_s[i]:.1f}s | params={params_M[i]:.2f}M | lat={lat_ms[i]:.6f} ms/img")

    # Radar plots: (a) base+super, (b) stack/oof
    def radar(ax, idxs, title):
        labels = ["Train Time","Model Complexity","Inference Speed"]
        angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
        angles += angles[:1]
        for i in idxs:
            vals = [score_train[i], score_params[i], score_speed[i]]
            vals += vals[:1]
            ax.plot(angles, vals, linewidth=1.1, alpha=0.9, label=names[i])
        ax.set_title(title, pad=14)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels)
        ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
        ax.set_ylim(0,1.0)

    base_idxs = [i for i,n in enumerate(names) if n.startswith(("concat_","attn_","gated_","super_"))]
    stack_idxs= [i for i,n in enumerate(names) if n.startswith(("stack_",))]

    fig = plt.figure(figsize=(11,10))
    ax1 = plt.subplot(2,1,1, polar=True)
    radar(ax1, base_idxs, "Base Heads + Super-Ensemble (normalized; higher better)")
    ax1.legend(loc="center left", bbox_to_anchor=(1.05, 0.5), fontsize=7, frameon=False)

    ax2 = plt.subplot(2,1,2, polar=True)
    radar(ax2, stack_idxs, "Stacked / OOF Methods (normalized; higher better)")
    ax2.legend(loc="center left", bbox_to_anchor=(1.05, 0.5), fontsize=7, frameon=False)

    plt.tight_layout()
    out_fig = os.path.join(OUT_DIR, "efficiency_radar.png")
    plt.savefig(out_fig, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", os.path.abspath(out_fig))
    print("Saved:", os.path.abspath(out_csv))

# ============================= MAIN ===========================================================
set_all_seeds(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
if USE_TEST2:
    os.makedirs(TEST2_CACHE, exist_ok=True)

print("\n" + "="*120)
print(f"TRAIN DATASET: {TRAIN_NAME}")
print(f"ROOT: {TRAIN_ROOT}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")
print("="*120)

# Split TRAIN_ROOT only (TEST2 is eval-only, never split)
split_data_into_folders_md5_grouped(TRAIN_ROOT, VAL_RATIO, TEST_RATIO, seed=SEED)

# Load splits
ds_train = RawImageFolder(os.path.join(TRAIN_ROOT,"train"))
ds_val   = RawImageFolder(os.path.join(TRAIN_ROOT,"val"))
ds_test  = RawImageFolder(os.path.join(TRAIN_ROOT,"test"))

class_names = list(ds_train.class_to_idx.keys())
num_classes = len(class_names)
print("TRAIN classes:", class_names)

with open(os.path.join(OUT_DIR, "class_names.txt"), "w", encoding="utf-8") as f:
    for c in class_names:
        f.write(c + "\n")

NONE_IDX = detect_none_index(class_names)
assert NONE_IDX is not None, f"Could not find NONE/No-tumor class in {class_names}. Rename it to include notumor/no_tumor/none."
tumor_classes = [class_names[i] for i in range(num_classes) if i != NONE_IDX]
print(f"[Detection] NONE_IDX={NONE_IDX} ({class_names[NONE_IDX]}) | Tumor classes={tumor_classes} | DETECT_THRESH={DETECT_THRESH:.2f}")

# TEST2 dataset (eval-only)
ds_test2 = None
if USE_TEST2:
    ds_test2 = build_external_test2_dataset(TEST2_ROOT, ds_train.class_to_idx)

# Choose models
MODELS = []
if INCLUDE_DINOV3_IF_AVAILABLE:
    DINOV3 = pick_dinov3()
    if DINOV3: MODELS.append(DINOV3)

for m in CAND_MODELS:
    if m == "tf_efficientnetv2_m_in21ft1k":
        eff = pick_effnet_fallback()
        if eff: MODELS.append(eff)
    else:
        MODELS.append(m)

print("Chosen models:", MODELS)

# Load backbones
BACKBONES=[]
for m in MODELS:
    try:
        bb = TimmBackbone(m); BACKBONES.append(bb)
        print(f"[Backbone] {m} | feat_dim={bb.out_dim} | input={bb.size}")
    except Exception as e:
        print(f"[WARN] Skipped {m}: {e}")
assert BACKBONES, "No backbones loaded."

bb_dims = [bb.out_dim for bb in BACKBONES]

# Cache features
cache_train = cache_features_for_dataset(ds_train, "train", "none", BACKBONES, CACHE_DIR)
cache_val   = cache_features_for_dataset(ds_val,   "val",   TTA_MODE_VALTEST, BACKBONES, CACHE_DIR)
cache_test  = cache_features_for_dataset(ds_test,  "test",  TTA_MODE_VALTEST, BACKBONES, CACHE_DIR)

cache_test2 = None
if ds_test2 is not None:
    cache_test2 = cache_features_for_dataset(ds_test2, "test2", TTA_MODE_VALTEST, BACKBONES, TEST2_CACHE)

# Build streams
Xtr_full, fp_tr = build_streams(cache_train, BACKBONES)
Xva_full, fp_va = build_streams(cache_val,   BACKBONES)
Xte_full, fp_te = build_streams(cache_test,  BACKBONES)

X2_full, fp2 = (None, None)
if cache_test2 is not None:
    X2_full, fp2 = build_streams(cache_test2, BACKBONES)

ytr = labels_for_folder(os.path.join(TRAIN_ROOT,"train"))
yva = labels_for_folder(os.path.join(TRAIN_ROOT,"val"))
yte = labels_for_folder(os.path.join(TRAIN_ROOT,"test"))
y2  = torch.tensor(ds_test2.labels, dtype=torch.long) if ds_test2 is not None else None

# Z-score (fit only on TRAIN)
if USE_ZSCORE:
    mu_full, sd_full = fit_zscore(Xtr_full)
    Xtr_full = apply_zscore(Xtr_full, mu_full, sd_full)
    Xva_full = apply_zscore(Xva_full, mu_full, sd_full)
    Xte_full = apply_zscore(Xte_full, mu_full, sd_full)
    if X2_full is not None:
        X2_full = apply_zscore(X2_full, mu_full, sd_full)

    if USE_FINGERPRINT and fp_tr is not None:
        mu_fp, sd_fp = fit_zscore(fp_tr)
        fp_tr = apply_zscore(fp_tr, mu_fp, sd_fp)
        fp_va = apply_zscore(fp_va, mu_fp, sd_fp)
        fp_te = apply_zscore(fp_te, mu_fp, sd_fp)
        if fp2 is not None:
            fp2 = apply_zscore(fp2, mu_fp, sd_fp)

print("Stream shapes:",
      "\n  train:", Xtr_full.shape,
      "\n  val  :", Xva_full.shape,
      "\n  test :", Xte_full.shape,
      "\n  test2:", (X2_full.shape if X2_full is not None else None))
if fp_tr is not None:
    print("FP shapes:",
          "\n  train:", fp_tr.shape,
          "\n  val  :", fp_va.shape,
          "\n  test :", fp_te.shape,
          "\n  test2:", (fp2.shape if fp2 is not None else None))

# ============================= RUN 28 VARIANTS ===============================================
results = []
base_records = []

# 12 base variants: 3 heads × (fp/nofp) × (single/ens)
base_specs=[]
for head_type in ["concat","attn","gated"]:
    for use_fp in [True, False]:
        base_specs.append((f"{head_type}_{'fp' if use_fp else 'nofp'}_single", head_type, use_fp, "single"))
        base_specs.append((f"{head_type}_{'fp' if use_fp else 'nofp'}_ens",    head_type, use_fp, "ens"))

for nm, head_type, use_fp, mode in base_specs:
    rec = evaluate_variant(
        nm, head_type, use_fp, mode,
        Xtr_full, Xva_full, Xte_full, X2_full,
        fp_tr, fp_va, fp_te, fp2,
        ytr, yva, yte, y2,
        bb_dims, num_classes, class_names, NONE_IDX,
        OUT_DIR
    )
    results.append(rec)
    base_records.append(rec)

# 1 super-ensemble (best SINGLE per head_type)
super_res = evaluate_super_ensemble(base_records, yva, yte, y2, NONE_IDX, OUT_DIR, name="super_bestheads")
results.append(super_res)


# ============================= AUTO-PICK STACK CONFIGS (VAL-only, no leakage) =============================
# Paste this CELL 1 right after:
#   results.append(super_res)
# Then paste CELL 2 after this cell.

# Pick metric for selecting top heads (VAL only)
STACK_PICK_METRIC = "macroF1"   # "macroF1" or "cls_acc" (use what you trust most)
PREFER_MODE = "ens"             # prefer *_ens variants when choosing bases

def _stack_score(r, metric=STACK_PICK_METRIC):
    try:
        v = r.get("val", {})
        s = v.get(metric, None)
        if s is None:
            s = v.get("cls_acc", None)
        return float(s) if s is not None else float("-inf")
    except Exception:
        return float("-inf")

def _base_candidates(results_list):
    out = []
    for r in results_list:
        if not isinstance(r, dict): 
            continue
        if r.get("head_type") not in ("concat","attn","gated"): 
            continue
        if r.get("mode") not in ("single","ens"):
            continue
        if "best_val_logits_raw" not in r:
            continue
        out.append(r)
    return out

def _pick_top_k_unique_headtype(cands, k, use_fp=None, prefer_mode=PREFER_MODE):
    # filter by fp flag
    if use_fp is not None:
        cands = [r for r in cands if bool(r.get("use_fp", False)) == bool(use_fp)]

    # sort: prefer mode + score
    def keyfun(r):
        return (1 if r.get("mode")==prefer_mode else 0, _stack_score(r))

    cands = sorted(cands, key=keyfun, reverse=True)

    picked = []
    used_head = set()
    for r in cands:
        ht = r.get("head_type")
        if ht in used_head:
            continue
        picked.append(r)
        used_head.add(ht)
        if len(picked) == k:
            break

    if len(picked) < k:
        raise ValueError(f"Not enough base variants to pick k={k} (use_fp={use_fp}). Got {len(picked)}")
    return picked

def _pick_top3_mixed(cands, prefer_mode=PREFER_MODE):
    # try to include at least one fp and one nofp if possible
    best_all = sorted(cands, key=lambda r: (1 if r.get("mode")==prefer_mode else 0, _stack_score(r)), reverse=True)

    # unique by head_type
    def unique_by_head(lst):
        out=[]; used=set()
        for r in lst:
            ht=r["head_type"]
            if ht in used: 
                continue
            out.append(r); used.add(ht)
        return out

    best_all = unique_by_head(best_all)

    fp = [r for r in best_all if bool(r.get("use_fp", False))]
    nf = [r for r in best_all if not bool(r.get("use_fp", False))]

    picked=[]
    if fp and nf:
        picked.append(best_all[0])  # best overall
        # force opposite fpness
        opp = nf if bool(picked[0].get("use_fp", False)) else fp
        picked.append(opp[0])
        # third best remaining
        used = {(r["head_type"], bool(r.get("use_fp", False))) for r in picked}
        for r in best_all:
            kk = (r["head_type"], bool(r.get("use_fp", False)))
            if kk not in used:
                picked.append(r)
                break
    else:
        picked = best_all[:3]

    # ensure 3
    if len(picked) < 3:
        used = {(r["head_type"], bool(r.get("use_fp", False))) for r in picked}
        for r in best_all:
            kk = (r["head_type"], bool(r.get("use_fp", False)))
            if kk not in used:
                picked.append(r); used.add(kk)
            if len(picked)==3:
                break

    if len(picked) != 3:
        raise ValueError("Could not pick 3 mixed variants.")
    return picked

def _names(recs): 
    return [r["name"] for r in recs]

base_cands = _base_candidates(results)

# Pick top2/top3 for fp/nofp and mixed top3
have_fp = any(bool(r.get("use_fp", False)) for r in base_cands)

top2_fp   = _pick_top_k_unique_headtype(base_cands, 2, use_fp=True)  if have_fp else None
top3_fp   = _pick_top_k_unique_headtype(base_cands, 3, use_fp=True)  if have_fp else None
top2_nofp = _pick_top_k_unique_headtype(base_cands, 2, use_fp=False)
top3_nofp = _pick_top_k_unique_headtype(base_cands, 3, use_fp=False)
top3_mix  = _pick_top3_mixed(base_cands)

print("\n" + "-"*120)
print("[AUTO-PICK] metric:", STACK_PICK_METRIC, "| prefer mode:", PREFER_MODE)
if have_fp:
    print("[AUTO-PICK] FP   top2:", _names(top2_fp))
    print("[AUTO-PICK] FP   top3:", _names(top3_fp))
else:
    print("[AUTO-PICK] FP not available in base variants -> will fallback to NOFP picks for FP stacks.")
print("[AUTO-PICK] NOFP top2:", _names(top2_nofp))
print("[AUTO-PICK] NOFP top3:", _names(top3_nofp))
print("[AUTO-PICK] MIX  top3:", _names(top3_mix))
print("-"*120)

# ============================= 5 VAL-driven logistic stacks (AUTO) =============================
if have_fp:
    results.append(evaluate_stacking_val_driven(
        results, yva, yte, y2, NONE_IDX,
        name="stack_lr_full_top2_fp",
        base_variants=_names(top2_fp), kfold=5
    ))
    results.append(evaluate_stacking_val_driven(
        results, yva, yte, y2, NONE_IDX,
        name="stack_lr_full_top3_fp",
        base_variants=_names(top3_fp), kfold=5
    ))
else:
    # fallback (keeps count = 28)
    results.append(evaluate_stacking_val_driven(
        results, yva, yte, y2, NONE_IDX,
        name="stack_lr_full_top2_fp",
        base_variants=_names(top2_nofp), kfold=5
    ))
    results.append(evaluate_stacking_val_driven(
        results, yva, yte, y2, NONE_IDX,
        name="stack_lr_full_top3_fp",
        base_variants=_names(top3_nofp), kfold=5
    ))

results.append(evaluate_stacking_val_driven(
    results, yva, yte, y2, NONE_IDX,
    name="stack_lr_full_top2_nofp",
    base_variants=_names(top2_nofp), kfold=5
))
results.append(evaluate_stacking_val_driven(
    results, yva, yte, y2, NONE_IDX,
    name="stack_lr_full_top3_nofp",
    base_variants=_names(top3_nofp), kfold=5
))
results.append(evaluate_stacking_val_driven(
    results, yva, yte, y2, NONE_IDX,
    name="stack_lr_full_top3_mixed",
    base_variants=_names(top3_mix), kfold=5
))

# Build lookup for fair inference cost (used by OOF stacks)
cost_lookup = {
    r["name"]: {"train_time_s": r.get("train_time_s",0.0), "params": r.get("params",0), "latency_ms": r.get("latency_ms",0.0)}
    for r in results if isinstance(r, dict) and "name" in r
}

# Convert selected base recs into head_specs for TRAIN-only OOF stacking
def _to_head_specs(recs):
    return tuple((r["head_type"], bool(r.get("use_fp", False))) for r in recs)

oof_top2_fp_specs   = _to_head_specs(top2_fp)   if have_fp else _to_head_specs(top2_nofp)
oof_top3_fp_specs   = _to_head_specs(top3_fp)   if have_fp else _to_head_specs(top3_nofp)
oof_top2_nofp_specs = _to_head_specs(top2_nofp)
oof_top3_nofp_specs = _to_head_specs(top3_nofp)
oof_top3_mix_specs  = _to_head_specs(top3_mix)

print("\n[OOF head_specs]")
print("  top2_fp  :", oof_top2_fp_specs)
print("  top3_fp  :", oof_top3_fp_specs)
print("  top2_nofp:", oof_top2_nofp_specs)
print("  top3_nofp:", oof_top3_nofp_specs)
print("  top3_mix :", oof_top3_mix_specs)


# ============================= 10 TRAIN-only OOF stacking variants (AUTO) =============================

results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top2_fp_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top2_single_fp", within_head_ensemble=False,
    base_cost_lookup=cost_lookup
))
results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top2_fp_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top2_ens_fp", within_head_ensemble=True,
    base_cost_lookup=cost_lookup
))

results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top3_fp_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top3_single_fp", within_head_ensemble=False,
    base_cost_lookup=cost_lookup
))
results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top3_fp_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top3_ens_fp", within_head_ensemble=True,
    base_cost_lookup=cost_lookup
))

results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top2_nofp_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top2_single_nofp", within_head_ensemble=False,
    base_cost_lookup=cost_lookup
))
results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top2_nofp_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top2_ens_nofp", within_head_ensemble=True,
    base_cost_lookup=cost_lookup
))

results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top3_nofp_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top3_single_nofp", within_head_ensemble=False,
    base_cost_lookup=cost_lookup
))
results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top3_nofp_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top3_ens_nofp", within_head_ensemble=True,
    base_cost_lookup=cost_lookup
))

results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top3_mix_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top3_single_mixed", within_head_ensemble=False,
    base_cost_lookup=cost_lookup
))
results.append(evaluate_stacking_train_oof(
    Xtr_full, Xva_full, Xte_full, X2_full,
    fp_tr, fp_va, fp_te, fp2,
    ytr, yva, yte, y2,
    bb_dims, num_classes, NONE_IDX,
    head_specs=oof_top3_mix_specs,
    kfold=5, inner_val_frac=0.10, name="stack_oof_full_top3_ens_mixed", within_head_ensemble=True,
    base_cost_lookup=cost_lookup
))

assert len(results) == 28, f"Expected 28 variants, got {len(results)}"

# ============================= SUMMARY CSV + PRINT ===========================================
def _fmt_ci(ci): 
    return f"[{ci[0]:.4f},{ci[1]:.4f}]"

headers = [
    "name","T",
    "VAL_cls_acc","VAL_cls_acc_CI","VAL_macroF1","VAL_macroAUROC","VAL_macroAP","VAL_ECE","VAL_Brier",
    "VAL_det_acc","VAL_tumor_subtype_acc","VAL_pipeline_acc",
    "TEST_cls_acc","TEST_cls_acc_CI","TEST_macroF1","TEST_macroAUROC","TEST_macroAP","TEST_ECE","TEST_Brier",
    "TEST_det_acc","TEST_tumor_subtype_acc","TEST_pipeline_acc",
    "TEST2_cls_acc","TEST2_cls_acc_CI","TEST2_macroF1","TEST2_macroAUROC","TEST2_macroAP","TEST2_ECE","TEST2_Brier",
    "TEST2_det_acc","TEST2_tumor_subtype_acc","TEST2_pipeline_acc",
]

rows=[]
for r in results:
    v = r["val"]; t = r["test"]; t2 = r.get("test2", None)

    def pack(m):
        if m is None:
            return ["NA"]*11
        return [
            f"{m['cls_acc']:.4f}", _fmt_ci(m["cls_acc_CI"]),
            f"{m['macroF1']:.4f}", f"{m['macroAUROC']:.4f}", f"{m['macroAP']:.4f}",
            f"{m['ECE']:.4f}", f"{m['Brier']:.4f}",
            f"{m['det_acc']:.4f}", f"{m['tumor_subtype_acc']:.4f}", f"{m['pipeline_acc']:.4f}",
        ]

    row = [r["name"], f"{float(r.get('T',1.0)):.2f}"]
    row += pack(v)
    row += pack(t)
    row += pack(t2)
    rows.append(row)

csv_path = os.path.join(OUT_DIR, "ablation_summary.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(headers); w.writerows(rows)

print("\n=== Ablation Summary (VAL / TEST / TEST2) ===")
print(tabulate(rows, headers=headers, tablefmt="github"))

best = max(results, key=lambda r: r["val"]["cls_acc"])
print(f"\n[Best by VAL classification accuracy] {best['name']} | T={best.get('T',1.0):.2f} | VAL_cls_acc={best['val']['cls_acc']:.4f} | TEST_cls_acc={best['test']['cls_acc']:.4f}")

print("Saved:", os.path.abspath(csv_path))
print("Outputs directory:", os.path.abspath(OUT_DIR))

# ============================= EXPORT ALL 28 + Efficiency ====================================
export_predictions_for_all(results, OUT_DIR, ds_test, yte, class_names, ds_test2, y2)
make_efficiency_tables_and_radar(results, OUT_DIR)

# cleanup
del BACKBONES, cache_train, cache_val, cache_test
if cache_test2 is not None: del cache_test2
torch.cuda.empty_cache()
